In [2]:
import matplotlib.pyplot as plt
# import contextily as ctx
import pandas as pd
import geopandas as gpd
from shapely import wkt
from scipy.stats import entropy
import folium

### **1. Prepare GeoJSON data**

In [3]:
path = "/content/drive/MyDrive/CUSP-GX 7103 CAPSTONE/CAPSTONE WORK PROGRESS/Output/GeoJSON/"

borough = gpd.read_file(path + "London_Borough.geojson")
ward = gpd.read_file(path + "London_Ward.geojson")
cluster_points = gpd.read_file(path + "London_maz_hdbscan.geojson")
cluster_convexhulls = gpd.read_file(path + "London_maz_convexhulls.geojson")

In [4]:
print(borough.columns)
print(ward.columns)

Index(['OBJECTID', 'lad22cd', 'Shape_Length', 'Shape_Area', 'geometry'], dtype='object')
Index(['FID', 'NAME', 'GSS_CODE', 'DISTRICT', 'LAGSSCODE', 'HECTARES',
       'NONLD_AREA', 'geometry'],
      dtype='object')


### **2. HDBSCAN Interactive Map**

In [ ]:
import matplotlib
import matplotlib.colors as colors
import colorsys # Import colorsys for HLS color manipulation
import random # Import random for shuffling colors

m = folium.Map(
    location=[51.509865, -0.118092],   # Central London
    zoom_start=11,
    tiles="CartoDB PositronNoLabels"
)

# Define a style function for Boroughs (cool tone)
def borough_style_function(feature):
    return {
        'fillColor': 'rgba(173,216,230,0.4)', # Light blue, semi-transparent
        'color': 'rgba(70,130,180,0.6)', # Steel blue border, slightly less transparent
        'weight': 0.5,
        'fillOpacity': 0.4,
        'opacity': 0.6
    }

# Define a style function for Wards (another cool tone)
def ward_style_function(feature):
    return {
        'fillColor': 'rgba(152,251,152,0.4)', # Pale green, semi-transparent
        'color': 'rgba(60,179,113,0.6)', # Medium sea green border, slightly less transparent
        'weight': 0.5,
        'fillOpacity': 0.4,
        'opacity': 0.6
    }

# Define a highlight function for all layers
def highlight_function(x):
    return {
        'fillOpacity': 0.7,
        'weight': 3,
        'color': 'red'
    }

# Add Borough GeoDataFrame as a GeoJson layer with tooltip, highlight, and layer control
borough_tooltip = folium.features.GeoJsonTooltip(fields=['lad22cd'], aliases=['Borough Code:'])
folium.GeoJson(
    borough,
    name='London Boroughs', # Name for LayerControl
    style_function=borough_style_function,
    highlight_function=highlight_function, # Add highlight function
    tooltip=borough_tooltip
).add_to(m)

# Add Ward GeoDataFrame as a GeoJson layer with tooltip, highlight, and layer control
folium.GeoJson(
    ward,
    name='London Wards', # Name for LayerControl
    style_function=ward_style_function,
    highlight_function=highlight_function, # Add highlight function
    tooltip=folium.features.GeoJsonTooltip(fields=['NAME'], aliases=['Ward Name:'])
).add_to(m)

# Get unique cluster labels and create a color map
unique_clusters = sorted(cluster_convexhulls['cluster_label'].unique())
num_clusters = len(unique_clusters)

# Use a warm and visually distinct colormap for clusters, with desaturation for Morandi effect
if num_clusters > 0:
    color_map = matplotlib.colormaps.get_cmap('Set3') # Good for distinct colors
    saturation_factor = 0.7 # Adjust this value (0 to 1, 1 is original, 0 is grayscale) for Morandi effect

    # Generate all desaturated colors first
    generated_colors = []
    for i in range(num_clusters):
        rgba_orig = color_map(i / (num_clusters - 1) if num_clusters > 1 else 0)
        r_orig, g_orig, b_orig, a_orig = rgba_orig

        # Convert RGB to HLS to easily manipulate saturation
        h, l, s_orig = colorsys.rgb_to_hls(r_orig, g_orig, b_orig)

        # Desaturate
        s_new = s_orig * saturation_factor

        # Convert HLS back to RGB
        r_new, g_new, b_new = colorsys.hls_to_rgb(h, l, s_new)
        generated_colors.append(colors.to_hex([r_new, g_new, b_new]))

    # Shuffle the generated colors to increase randomness in assignment
    random.shuffle(generated_colors)

    # Assign shuffled colors to cluster labels
    cluster_colors = {label: generated_colors[i] for i, label in enumerate(unique_clusters)}
else:
    cluster_colors = {} # Handle case with no clusters

# Define a style function for cluster convex hulls
def cluster_style_function(feature):
    cluster_label = feature['properties']['cluster_label']
    return {
        'fillColor': cluster_colors.get(cluster_label, '#808080'), # Default to grey if label not found
        'color': cluster_colors.get(cluster_label, '#808080'), # Border color same as fill
        'weight': 1,
        'fillOpacity': 1.0, # More opaque fill - changed from 0.7 to 1.0
        'opacity': 1.0 # Fully opaque border
    }

# Add Cluster Convex Hulls as a GeoJson layer with tooltip, highlight, and layer control
folium.GeoJson(
    cluster_convexhulls,
    name='Cluster Convex Hulls', # Name for LayerControl
    style_function=cluster_style_function,
    highlight_function=highlight_function, # Add highlight function
    tooltip=folium.features.GeoJsonTooltip(fields=['cluster_label'], aliases=['Cluster Label:'])
).add_to(m)

# Add Layer Control to the map
folium.LayerControl().add_to(m)

print("Folium map initialized successfully with optimized layers including distinct cool/warm tones and highlight function.")

Folium map initialized successfully with optimized layers including distinct cool/warm tones and highlight function.


In [ ]:
m

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
output_file = 'london_map.html'
m.save(output_file)
print(f"Map saved to {output_file}")

Map saved to london_map.html
